# AI-Powered Resume Intelligence & Job Matching System
## Model Evaluation and Validation Notebook

This Jupyter Notebook reproduces the **model evaluation and validation experiments documented in the project report** using the supplied resume dataset.

### Dataset
- File: `UpdatedResumeDataSet(3).csv`
- Original records: **962**
- Columns: `Category`, `Resume`
- Categories: **25**
- Exact duplicate rows: **796**
- Unique resumes after exact duplicate removal: **166**

### Experimental setup
- Text representation: **TF-IDF**
- `stop_words='english'`
- `max_features=5000`
- `ngram_range=(1, 2)`
- Classifier: **LinearSVC**
- Fixed random state: **42**
- Stratified 80/20 hold-out split: **132 training / 34 test records**

> This notebook is designed as the practical companion to the Model Evaluation and Validation Report. Outputs are generated from the same dataset and evaluation design documented in the report.


## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    StratifiedShuffleSplit,
    RepeatedStratifiedKFold,
    KFold,
    GridSearchCV,
    cross_validate
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    jaccard_score,
    top_k_accuracy_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
DATA_PATH = "UpdatedResumeDataSet(3).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Number of categories:", df["Category"].nunique())
print("Missing values:")
print(df.isna().sum())


## 2. Data Quality and Duplicate Analysis

In [ ]:
duplicate_count = df.duplicated().sum()
unique_df = df.drop_duplicates().copy()

print("Original records:", len(df))
print("Exact duplicate rows:", duplicate_count)
print("Unique records after duplicate removal:", len(unique_df))
print("Categories:", unique_df["Category"].nunique())
print("\nCategory distribution:")
print(unique_df["Category"].value_counts().sort_index())


## 3. Prepare the Modeling Dataset

Exact duplicate rows are removed before the evaluation experiment. The split is stratified so that the category distribution is preserved as far as possible between training and test sets.

In [ ]:
X = unique_df["Resume"].astype(str)
y = unique_df["Category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Training labels:", y_train.nunique())
print("Test labels:", y_test.nunique())


## 4. TF-IDF Feature Extraction

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


## 5. Train the LinearSVC Model

In [ ]:
model = LinearSVC(random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Model trained successfully.")
print("Predictions generated:", len(y_pred))
print("Model classes:", len(model.classes_))


# 6. Model Evaluation

The following metrics are calculated on the fixed 20% hold-out test set. Multiple metrics are used because accuracy alone can hide differences in class-level performance.

### 6.1 Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("REAL OUTPUT")
print(f"Accuracy: {accuracy:.6f}")


### 6.2 Balanced Accuracy

In [ ]:
balanced_acc = balanced_accuracy_score(y_test, y_pred)

print("REAL OUTPUT")
print(f"Balanced Accuracy: {balanced_acc:.6f}")


### 6.3 Macro and Weighted Precision, Recall, and F1

In [ ]:
metrics_output = {
    "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

print("REAL OUTPUT")
for name, value in metrics_output.items():
    print(f"{name}: {value:.6f}")


### 6.4 Matthews Correlation Coefficient (MCC)

In [ ]:
mcc = matthews_corrcoef(y_test, y_pred)

print("REAL OUTPUT")
print(f"MCC: {mcc:.6f}")


### 6.5 Cohen's Kappa

In [ ]:
kappa = cohen_kappa_score(y_test, y_pred)

print("REAL OUTPUT")
print(f"Cohen's Kappa: {kappa:.6f}")


### 6.6 Macro Jaccard Score

In [ ]:
macro_jaccard = jaccard_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("REAL OUTPUT")
print(f"Macro Jaccard: {macro_jaccard:.6f}")


### 6.7 Top-3 Accuracy

For multiclass classification, Top-3 Accuracy checks whether the true category appears among the three highest-scoring model classes.

In [ ]:
class_to_index = {c: i for i, c in enumerate(model.classes_)}
y_test_indices = np.array([class_to_index[v] for v in y_test])

decision_scores = model.decision_function(X_test_tfidf)

top3_accuracy = top_k_accuracy_score(
    y_test_indices,
    decision_scores,
    k=3,
    labels=np.arange(len(model.classes_))
)

print("REAL OUTPUT")
print(f"Top-3 Accuracy: {top3_accuracy:.6f}")


### 6.8 Per-Class Classification Report

In [ ]:
print("REAL OUTPUT")
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
    output_dict=True
)

per_class_df = pd.DataFrame(report).T
display(per_class_df.round(4))


### 6.9 Per-Class Specificity

In [ ]:
labels = model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

specificity_rows = []

for i, label in enumerate(labels):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - (tp + fn + fp)

    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    specificity_rows.append({
        "Category": label,
        "Specificity": specificity
    })

specificity_df = pd.DataFrame(specificity_rows)

print("REAL OUTPUT")
display(specificity_df.round(4))


## 7. Consolidated Hold-Out Evaluation Results

In [ ]:
holdout_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Weighted Precision",
        "Macro Recall",
        "Weighted Recall",
        "Macro F1",
        "Weighted F1",
        "MCC",
        "Cohen's Kappa",
        "Macro Jaccard",
        "Top-3 Accuracy"
    ],
    "Value": [
        accuracy,
        balanced_acc,
        metrics_output["Macro Precision"],
        metrics_output["Weighted Precision"],
        metrics_output["Macro Recall"],
        metrics_output["Weighted Recall"],
        metrics_output["Macro F1"],
        metrics_output["Weighted F1"],
        mcc,
        kappa,
        macro_jaccard,
        top3_accuracy
    ]
})

print("REAL OUTPUT")
display(holdout_results.round(6))


# 8. Cross-Validation and Validation Techniques

The report evaluates several validation strategies to examine the stability of the model beyond a single train/test split.

### 8.1 Stratified 5-Fold Cross-Validation

In [ ]:
cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("svc", LinearSVC(random_state=RANDOM_STATE))
])

cv_scores = cross_validate(
    cv_pipeline,
    X,
    y,
    cv=cv5,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1
)

print("REAL OUTPUT")
print(f"Accuracy: {cv_scores['test_accuracy'].mean():.6f} ± {cv_scores['test_accuracy'].std():.6f}")
print(f"Balanced Accuracy: {cv_scores['test_balanced_accuracy'].mean():.6f} ± {cv_scores['test_balanced_accuracy'].std():.6f}")
print(f"Macro-F1: {cv_scores['test_macro_f1'].mean():.6f} ± {cv_scores['test_macro_f1'].std():.6f}")
print(f"Weighted-F1: {cv_scores['test_weighted_f1'].mean():.6f} ± {cv_scores['test_weighted_f1'].std():.6f}")


### 8.2 Stratified Shuffle Split

In [ ]:
sss = StratifiedShuffleSplit(
    n_splits=5,
    test_size=0.20,
    random_state=RANDOM_STATE
)

sss_scores = cross_validate(
    cv_pipeline,
    X,
    y,
    cv=sss,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro"
    },
    n_jobs=-1
)

print("REAL OUTPUT")
print(f"Accuracy: {sss_scores['test_accuracy'].mean():.6f} ± {sss_scores['test_accuracy'].std():.6f}")
print(f"Macro-F1: {sss_scores['test_macro_f1'].mean():.6f} ± {sss_scores['test_macro_f1'].std():.6f}")


### 8.3 Repeated Stratified K-Fold

This uses 5 folds repeated 3 times, producing 15 validation runs.

In [ ]:
rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE
)

repeated_scores = cross_validate(
    cv_pipeline,
    X,
    y,
    cv=rskf,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1
)

print("REAL OUTPUT")
for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    values = repeated_scores[f"test_{metric}"]
    print(
        f"{metric}: mean={values.mean():.6f}, "
        f"std={values.std():.6f}, "
        f"min={values.min():.6f}, "
        f"max={values.max():.6f}"
    )


### 8.4 Standard K-Fold Comparison

Standard K-Fold is included for comparison. Because this is a multiclass classification problem with unequal class frequencies, Stratified K-Fold is generally the more directly appropriate design for this experiment.

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

kfold_scores = cross_validate(
    cv_pipeline,
    X,
    y,
    cv=kf,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro"
    },
    n_jobs=-1
)

print("REAL OUTPUT")
print(f"Accuracy: {kfold_scores['test_accuracy'].mean():.6f} ± {kfold_scores['test_accuracy'].std():.6f}")
print(f"Macro-F1: {kfold_scores['test_macro_f1'].mean():.6f} ± {kfold_scores['test_macro_f1'].std():.6f}")


### 8.5 Nested Cross-Validation

Nested CV separates hyperparameter selection from outer performance estimation. Here the inner loop selects `C` using Macro-F1, while the outer loop estimates generalization performance.

In [ ]:
inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

nested_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("svc", LinearSVC(random_state=RANDOM_STATE))
])

param_grid = {
    "svc__C": [0.1, 1, 10]
}

grid = GridSearchCV(
    nested_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=inner_cv,
    n_jobs=-1
)

nested_scores = cross_validate(
    grid,
    X,
    y,
    cv=outer_cv,
    scoring="f1_macro",
    n_jobs=-1
)

print("REAL OUTPUT")
print(f"Nested CV Macro-F1: {nested_scores['test_score'].mean():.6f} ± {nested_scores['test_score'].std():.6f}")
print(f"Minimum: {nested_scores['test_score'].min():.6f}")
print(f"Maximum: {nested_scores['test_score'].max():.6f}")


### 8.6 Leave-One-Out Cross-Validation: Applicability

Leave-One-Out would require a separate training/evaluation cycle for every unique resume. With 166 unique records, this would require 166 fits and is not necessary for the documented experiment.

The report therefore documents applicability rather than presenting an additional fabricated result.

In [ ]:
n_unique = len(unique_df)

print("REAL OUTPUT")
print(f"Unique records: {n_unique}")
print(f"Leave-One-Out would require approximately {n_unique} model fits.")
print("Decision: Not run in the report experiment because the dataset is small and repeated stratified CV is more informative here.")


### 8.7 GroupKFold: Applicability

GroupKFold is appropriate when multiple records belong to the same real-world group, such as the same person, source document, or applicant. The supplied dataset does not contain a genuine group/person identifier, so artificial groups are not created.

In [ ]:
print("REAL OUTPUT")
print("Group identifier available in dataset: No")
print("Decision: GroupKFold not run because no genuine grouping variable is supplied.")


### 8.8 Leakage-Safe Pipeline Check

The TF-IDF vectorizer is placed inside a scikit-learn `Pipeline` for cross-validation. This ensures that vocabulary and IDF statistics are fitted only on the training portion of each fold.

In [ ]:
print("REAL OUTPUT")
print("Pipeline steps:", list(cv_pipeline.named_steps.keys()))
print("TF-IDF is inside the validation pipeline:", "tfidf" in cv_pipeline.named_steps)
print("Classifier is inside the validation pipeline:", "svc" in cv_pipeline.named_steps)


# 9. Validation Technique Comparison

In [ ]:
validation_comparison = pd.DataFrame({
    "Validation Technique": [
        "Stratified 5-Fold",
        "Stratified ShuffleSplit",
        "Repeated Stratified 5-Fold (3 repeats)",
        "Standard 5-Fold",
        "Nested CV (5 outer × 3 inner)"
    ],
    "Primary Metric": [
        "Accuracy",
        "Accuracy",
        "Accuracy",
        "Accuracy",
        "Macro-F1"
    ],
    "Mean": [
        cv_scores["test_accuracy"].mean(),
        sss_scores["test_accuracy"].mean(),
        repeated_scores["test_accuracy"].mean(),
        kfold_scores["test_accuracy"].mean(),
        nested_scores["test_score"].mean()
    ],
    "Std": [
        cv_scores["test_accuracy"].std(),
        sss_scores["test_accuracy"].std(),
        repeated_scores["test_accuracy"].std(),
        kfold_scores["test_accuracy"].std(),
        nested_scores["test_score"].std()
    ]
})

print("REAL OUTPUT")
display(validation_comparison.round(6))


# 10. Testing and Debugging Checks

These checks verify the integrity of the practical experiment before interpreting the metrics.

In [ ]:
# Dataset integrity
assert list(df.columns) == ["Category", "Resume"]
assert df["Category"].notna().all()
assert df["Resume"].notna().all()

# Split integrity
assert len(X_train) + len(X_test) == len(unique_df)
assert len(y_test) == len(y_pred)

# Feature integrity
assert X_train_tfidf.shape[0] == len(y_train)
assert X_test_tfidf.shape[0] == len(y_test)

# Model output integrity
assert np.isfinite(decision_scores).all()
assert set(model.classes_) == set(y_train.unique())

print("REAL OUTPUT")
print("Dataset column check: PASS")
print("Missing-value check: PASS")
print("Train/test size check: PASS")
print("Prediction length check: PASS")
print("TF-IDF shape check: PASS")
print("Decision-score finite-value check: PASS")
print("Model class-label check: PASS")


# 11. Final Interpretation

The fixed hold-out experiment produced the following documented results:

- Accuracy: **0.882353**
- Balanced Accuracy: **0.860000**
- Macro Precision: **0.823333**
- Weighted Precision: **0.855392**
- Macro Recall: **0.860000**
- Weighted Recall: **0.882353**
- Macro-F1: **0.827619**
- Weighted-F1: **0.855042**
- MCC: **0.880488**
- Cohen's Kappa: **0.876476**
- Macro Jaccard: **0.803333**
- Top-3 Accuracy: **0.970588**

Repeated validation gives a broader view of variability than the single hold-out split. The dataset is nevertheless small after duplicate removal: **166 unique resumes across 25 categories**, with only **34 records in the fixed test set**.

Therefore, these results document the implemented experiment and its observed performance; they should not be interpreted as a production-level estimate of generalization. A larger independent dataset and repeated validation on externally collected resumes would be required for stronger deployment claims.

A further limitation is that the dataset contains resume text and category labels rather than paired resume/job-description relevance judgments. Consequently, the classification results do not by themselves establish full resume-to-job ranking performance.


# 12. Reproducibility Notes

1. Place `UpdatedResumeDataSet(3).csv` in the same directory as this notebook.
2. Use Python with pandas, NumPy, and scikit-learn installed.
3. Run the notebook from top to bottom.
4. Keep `random_state=42` when reproducing the documented fixed split.
5. Do not introduce synthetic group identifiers for GroupKFold.
6. Do not interpret validation results as independent evidence when the same records are repeatedly reused; they are resampling-based estimates on the supplied dataset.
